## Extracting and cleaning data with DuckDB and Pandas


In [ ]:
import pandas as pd

pd.set_option("display.max_columns", None)  # show all cols
pd.set_option("display.max_colwidth", None)  # show full width of showing cols
pd.set_option(
    "display.expand_frame_repr", False
)  # print cols side by side as it's supposed to be

In [ ]:
# Nous commencons par importer les librairies nécessaires pour l'analyse des données.

import duckdb

ODIS_DUCKDB_FILE = "odis.duckdb"
PCC_DUCKDB_FILE = "dev.duckdb"

con = duckdb.connect(database=PCC_DUCKDB_FILE, read_only=True)
con.sql(f"ATTACH '{ODIS_DUCKDB_FILE}' AS odis;")

In [ ]:
# informacion de comunas y riesgos
query_catnat = """
SELECT *
  FROM dev.main.catnat_gaspar
  ORDER BY cod_commune ASC;
"""

cat_nat_2000 = con.sql(query_catnat)
cat_nat_2000_df = cat_nat_2000.df()

query_odis1 = """SELECT * FROM odis.gold_gold_com_dep_reg
ORDER BY CODGEO ASC;"""
odis_geolocalisation = con.sql(query_odis1).df()
odis_geolocalisation.head(2)

query_odis2 = """SELECT * FROM odis.gold_gold_emploi_demandeur
ORDER BY codgeo ASC;
"""
odis_emploi_demandeur = con.sql(query_odis2).df()
odis_emploi_demandeur["codgeo"] = odis_emploi_demandeur["codgeo"].str.zfill(5)


query_odis3 = """SELECT * FROM odis.gold_gold_typologies_territoires
ORDER BY codgeo ASC;"""
odis_topology = con.sql(query_odis3).df()


query_odis4 = """SELECT * FROM odis.gold_gold_population_taux_pauvrete
ORDER BY codgeo ASC;"""
odis_population_taux_pauvrete = con.sql(query_odis4).df()


In [ ]:
# Cleaning Climate data
from datetime import datetime

cat_nat_2000_df.head(10)
climate_data = pd.DataFrame(
    {
        "lib_commune": cat_nat_2000_df["lib_commune"],
        "cod_commune": cat_nat_2000_df["cod_commune"],
        "climate_risk": cat_nat_2000_df["lib_risque_jo"],
        "durée événement": pd.to_datetime(cat_nat_2000_df["dat_fin"], format="mixed")
        - pd.to_datetime(cat_nat_2000_df["dat_deb"], format="mixed"),
    }
)
df_climate = (
    climate_data.groupby(["cod_commune", "lib_commune", "climate_risk"])
    .agg(duree_total=("durée événement", "sum"), nb_events=("durée événement", "count"))
    .reset_index()
)
df_climate

In [ ]:
# Cleaning Socio-economic data
df_demandeur = (
    odis_emploi_demandeur.groupby("codgeo")
    .agg(nb_demandeurs=("Demandeurs_Emploi", "mean"))
    .reset_index()
)
odis_population_taux_pauvrete.fillna(0, inplace=True)
odis_population_taux_pauvrete["codgeo"] = (
    odis_population_taux_pauvrete["codgeo"].astype(str).str.zfill(5)
)

df_pauvrete = (
    odis_population_taux_pauvrete.groupby("codgeo")
    .agg(TP40=("TP40", "mean"), TP50=("TP50", "mean"), TP60=("TP60", "mean"))
    .reset_index()
)
merged = pd.merge(df_demandeur, df_pauvrete, left_on="codgeo", right_on="codgeo")
df_socio = pd.merge(
    merged, odis_topology, left_on="codgeo", right_on="codgeo"
).reset_index(drop=True)
df_socio.head(5)

In [ ]:
# Merging both datasets
df_merged = pd.merge(df_climate, df_socio, left_on="cod_commune", right_on="codgeo")
df_merged = df_merged.drop(columns=["codgeo", "lib_commune"])
df_merged.head(5)
df_merged["duree_total"] = df_merged["duree_total"].dt.total_seconds() / 60

In [ ]:
## Graphs

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

sns.lineplot(data=df_merged, x="nb_events", y="nb_demandeurs")

sns.regplot(
    data=df_merged,
    x="nb_events",
    y="nb_demandeurs",
    scatter=False,  # no puntos
    ci=False,  # sin intervalo de confianza
    label="Trend",
)

plt.legend()
plt.show()

In [ ]:
from sklearn.preprocessing import MinMaxScaler
import numpy as np

scaler = MinMaxScaler()
df_scaled = df_merged.copy()
df_scaled[["TP40", "duree_total"]] = scaler.fit_transform(
    df_merged[["TP40", "duree_total"]]
)

sns.lineplot(data=df_scaled, x="duree_total", y="TP40")
sns.regplot(
    data=df_scaled,
    x="duree_total",
    y="TP40",
    scatter=False,
    ci=False,
    label="Trend",
)

plt.legend()
plt.show()

In [ ]:
sns.lineplot(data=df_merged, x="nb_events", y="TP40")
sns.regplot(
    data=df_merged,
    x="nb_events",
    y="TP40",
    scatter=False,  # no puntos
    ci=False,  # sin intervalo de confianza
    label="Trend",
)

plt.legend()
plt.show()

In [ ]:
sns.lineplot(data=df_merged, x="nb_events", y="TP50")
sns.regplot(
    data=df_merged, x="nb_events", y="TP50", scatter=False, ci=True, label="Trend"
)

plt.legend()
plt.show()

In [ ]:
sns.lineplot(data=df_merged, x="nb_events", y="TP60")
sns.regplot(
    data=df_merged,
    x="nb_events",
    y="TP60",
    scatter=False,  # no puntos
    ci=False,  # sin intervalo de confianza
    label="Trend",
)

plt.legend()
plt.show()

In [ ]:
## Correlations

In [ ]:
# All correaltions done with different methods show a week correlation, lower than 0.1
corr1 = df_merged[["nb_events", "nb_demandeurs"]].corr(method="spearman")
print(corr1)

corr2 = df_merged[["nb_events", "nb_demandeurs"]].corr(method="pearson")
print(corr2)

corr3 = df_merged[["nb_events", "nb_demandeurs"]].corr(method="kendall")
print(corr3)

corr4 = df_merged[["nb_events", "TP60"]].corr(method="spearman")
print(corr4)

corr5 = df_merged[["nb_events", "TP60"]].corr(method="pearson")
print(corr5)

corr6 = df_merged[["nb_events", "TP60"]].corr(method="kendall")
print(corr6)


In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, mean_squared_error


X = df_merged[["nb_demandeurs", "TP40", "TP50", "TP60"]]
y = df_merged["nb_events"]


model = LinearRegression()
model.fit(X, y)

y_pred = model.predict(X)

r2 = r2_score(y, y_pred)
print("R^2:", r2)
mse = mean_squared_error(y, y_pred)
print("MSE:", mse)